# Toy Models of Superposition

**Thesis.** A nonlinear model given *sparse* inputs will represent more features than it
has dimensions — *superposition* — and sparsity is the knob that turns it on. The cost is
*interference*.

This notebook rebuilds the core of Elhage et al. (2022), *Toy Models of Superposition*,
following the paper's own build: a linear ceiling, then a ReLU model on dense data, then
we crank sparsity and watch five features arrange themselves as a pentagon in two
dimensions. Everything here is synthetic — no language model, no Arabic — because the
goal is to understand the *mechanism*.

## Act 0 — The puzzle

In real networks, single neurons are often *polysemantic*: one neuron responds to several
unrelated things. Why would a model do that instead of giving each concept its own
neuron? The paper's answer is **superposition** — when features are rare (sparse), a model
can pack more of them than it has dimensions by storing them in overlapping directions,
paying a little *interference* in return. We'll reproduce that from scratch and find the
exact knob — sparsity — that decides whether it happens.

In [ ]:
# lib: imports
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
print("torch", torch.__version__, "| seed", SEED)

## Act 1 — Features, and the linear ceiling

The paper models each feature as a direction in activation space. Our synthetic data
mirrors what it assumes real features look like: mostly-zero (sparse) and non-negative.
Each feature is zero with probability `S` (the sparsity) and otherwise uniform on
`[0, 1)`. Each feature also has an *importance* — a scalar weight on its share of the
loss — that decays across features, so the model has to make trade-offs.

In [ ]:
# lib: make_batch
def make_batch(n_features, sparsity, batch_size, generator=None):
    """Sparse synthetic features.

    Each entry is 0 with probability `sparsity`, otherwise uniform on [0, 1).
    Returns a tensor of shape [batch_size, n_features].
    """
    vals = torch.rand(batch_size, n_features, generator=generator)
    keep = torch.rand(batch_size, n_features, generator=generator) >= sparsity
    return vals * keep

In [ ]:
# lib: importance
def importance_weights(n_features, decay=0.7):
    """Importance Iᵢ = decay**i, shape [n_features]. Matches the paper's Iᵢ = 0.7^i."""
    return decay ** torch.arange(n_features, dtype=torch.float32)

In [ ]:
gen = torch.Generator().manual_seed(SEED)
demo = make_batch(n_features=5, sparsity=0.8, batch_size=8, generator=gen)
print("batch shape:", tuple(demo.shape), "| fraction nonzero:", (demo > 0).float().mean().item())
print("importance:", importance_weights(5).tolist())

### The model

We embed `n` features into `m < n` dimensions with a matrix `W` of shape `[m, n]` (column
`i` is feature `i`'s direction), then read them back out through `Wᵀ` and add a bias:

- **Linear model:** `x' = Wᵀ W x`. It can represent at most `m` features orthogonally —
  superposition is impossible by construction (`WᵀW` is invertible ⟺ no superposition).
- **ReLU model:** `x' = ReLU(Wᵀ W x + b)`. The non-linearity is what lets it tolerate the
  interference that superposition creates.

In [ ]:
# lib: toymodel
class ToyModel(torch.nn.Module):
    """Embed n features into m<n dims via W [m, n], read back through Wᵀ, add bias.

    forward: h = x @ W.T ; out = h @ W + b ; ReLU(out) if use_relu else out.
    """
    def __init__(self, n_features, n_hidden, use_relu=True):
        super().__init__()
        self.use_relu = use_relu
        self.W = torch.nn.Parameter(torch.empty(n_hidden, n_features))
        torch.nn.init.xavier_normal_(self.W)
        self.b = torch.nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        h = x @ self.W.T           # [B, m]
        out = h @ self.W + self.b  # [B, n]
        return F.relu(out) if self.use_relu else out

In [ ]:
m = ToyModel(n_features=5, n_hidden=2)
gen = torch.Generator().manual_seed(SEED)
xb = make_batch(5, sparsity=0.8, batch_size=4, generator=gen)
print("W:", tuple(m.W.shape), "| b:", tuple(m.b.shape), "| out:", tuple(m(xb).shape))

In [ ]:
# lib: train
def train(model, sparsity, importance, steps=10_000, lr=1e-3, batch_size=1024, seed=0):
    """Train `model` to reconstruct sparse features under importance-weighted MSE.

    Loss = mean over batch of Σᵢ importanceᵢ · (xᵢ − x'ᵢ)². Returns loss sampled every
    500 steps.
    """
    n_features = model.W.shape[1]
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        x = make_batch(n_features, sparsity, batch_size, generator=gen)
        out = model(x)
        loss = (importance * (x - out) ** 2).sum(dim=-1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 500 == 0:
            losses.append(loss.item())
    return losses

In [ ]:
# lib: feature_norms
def feature_norms(model):
    """L2 norm of each feature's embedding column (how strongly it is represented)."""
    return model.W.detach().norm(dim=0)

In [ ]:
imp5 = importance_weights(5)                       # Iᵢ = 0.7^i
lin = ToyModel(5, 2, use_relu=False)
train(lin, sparsity=0.0, importance=imp5, seed=SEED)
norms = feature_norms(lin)
print("linear / dense feature norms:", [round(v, 3) for v in norms.tolist()])
n_represented = int((norms > 0.5 * norms.max()).sum())
print("features clearly represented:", n_represented)
assert n_represented <= 2, "a 2-dim linear model should keep at most its top-2 features"

The linear model behaves like PCA: with two dimensions it keeps only the two most
important features and discards the rest. This is the ceiling — no non-linearity, so no
way to tolerate interference, so no superposition. Next we add the ReLU.

## Act 2 — Add the ReLU, keep the data dense

Does the ReLU alone create superposition? No. On dense data the ReLU model keeps the same
top-2 features the linear model did. The non-linearity only pays off once the data is
sparse — which is the whole point of Act 3.

In [ ]:
relu_dense = ToyModel(5, 2, use_relu=True)
train(relu_dense, sparsity=0.0, importance=imp5, seed=SEED)
norms = feature_norms(relu_dense)
print("ReLU / dense feature norms:", [round(v, 3) for v in norms.tolist()])
n_represented = int((norms > 0.5 * norms.max()).sum())
print("features clearly represented:", n_represented)
assert n_represented <= 2, "ReLU on DENSE data should still keep only its top-2 features"

Same story as the linear model: two features in, three discarded. The ReLU changed
nothing while the data is dense. Now we make the data sparse and watch it change
everything.

## Act 3 — Crank the sparsity: superposition appears

Now we sweep the sparsity `S` upward and retrain the ReLU model each time. As features get
rarer, the model stops throwing three of them away. It packs all five into two dimensions
by placing them in non-orthogonal directions — accepting some interference because sparse
features rarely collide. At `S = 0.9` the five directions form a **pentagon**.

In [ ]:
# lib: plot_features_2d
def plot_features_2d(W, ax=None, title=None):
    """Draw each column of a [2, n] weight matrix as a ray from the origin."""
    Wn = W.detach().cpu().numpy()
    ax = ax or plt.gca()
    for i in range(Wn.shape[1]):
        ax.plot([0.0, Wn[0, i]], [0.0, Wn[1, i]], marker="o")
    lim = float(np.abs(Wn).max()) * 1.1 + 1e-9
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.axhline(0, lw=0.5, color="gray"); ax.axvline(0, lw=0.5, color="gray")
    if title:
        ax.set_title(title)

In [ ]:
# lib: plot_WtW
def plot_WtW(W, ax=None, title=None):
    """Heatmap of WᵀW: diagonal = how strongly each feature is represented,
    off-diagonal = interference between features."""
    WtW = (W.T @ W).detach().cpu().numpy()
    ax = ax or plt.gca()
    im = ax.imshow(WtW, cmap="RdBu", vmin=-1.0, vmax=1.0)
    ax.set_xticks(range(WtW.shape[0])); ax.set_yticks(range(WtW.shape[0]))
    if title:
        ax.set_title(title)
    return im

In [ ]:
# lib: count_represented
def count_represented(model, frac=0.5):
    """How many feature columns have norm > frac * the largest column norm."""
    norms = feature_norms(model)
    return int((norms > frac * norms.max()).sum())

In [ ]:
sparsities = [0.0, 0.8, 0.9]
models = []
fig, axes = plt.subplots(2, len(sparsities), figsize=(4 * len(sparsities), 8))
for j, S in enumerate(sparsities):
    mdl = ToyModel(5, 2, use_relu=True)
    train(mdl, sparsity=S, importance=imp5, seed=SEED)
    models.append(mdl)
    plot_features_2d(mdl.W, ax=axes[0, j], title=f"S = {S}  ({count_represented(mdl)} represented)")
    plot_WtW(mdl.W, ax=axes[1, j], title=f"WᵀW  (S = {S})")
fig.suptitle("Features go from 2 orthogonal → antipodal pairs → a pentagon as sparsity rises")
fig.tight_layout()
plt.show()

reps = [count_represented(m) for m in models]
print("features represented at S =", sparsities, "→", reps)
assert reps[-1] > reps[0], "high sparsity should represent MORE features than dense"

The count of represented features climbs with sparsity — the existence proof. If a given
seed lands the sparse model in a local minimum (the paper notes these toys have
"energy-level" jumps and can get stuck), the exact geometry may not be a clean pentagon;
what matters is the direction of the effect — sparsity buys representation of more
features at the cost of the off-diagonal interference now visible in `WᵀW`.

## Act 4 — The sparsity phase, quantified

The pentagon is one seed at one size. To show the *effect* rather than an anecdote, we
move to the paper's demonstration size — `n = 20` features in `m = 5` dimensions,
`Iᵢ = 0.7^i` — and sweep sparsity over a grid, counting how many features end up
represented at each point. The count rises with sparsity: that curve is "sparsity drives
superposition" as a number, not a vibe.

In [ ]:
# lib: sparsity_sweep
def sparsity_sweep(n_features, n_hidden, sparsities, decay=0.7, seed=0):
    """Train one ReLU model per sparsity; return (counts_represented, [W per sparsity])."""
    imp = importance_weights(n_features, decay=decay)
    counts, weights = [], []
    for S in sparsities:
        mdl = ToyModel(n_features, n_hidden, use_relu=True)
        train(mdl, sparsity=S, importance=imp, seed=seed)
        counts.append(count_represented(mdl))
        weights.append(mdl.W.detach().clone())
    return counts, weights

In [ ]:
sweep_S = [0.0, 0.5, 0.7, 0.9, 0.97, 0.99]
counts, weights = sparsity_sweep(n_features=20, n_hidden=5, sparsities=sweep_S, seed=SEED)

fig, (ax_curve, ax_lo, ax_hi) = plt.subplots(1, 3, figsize=(15, 4))
ax_curve.plot(sweep_S, counts, marker="o")
ax_curve.set_xlabel("sparsity S"); ax_curve.set_ylabel("features represented (of 20)")
ax_curve.set_title("More sparsity → more features in superposition")
plot_WtW(weights[0], ax=ax_lo, title=f"WᵀW at S = {sweep_S[0]} (dense)")
plot_WtW(weights[-1], ax=ax_hi, title=f"WᵀW at S = {sweep_S[-1]} (sparse)")
fig.tight_layout()
plt.show()

print("features represented across S =", sweep_S, "→", counts)
assert counts[-1] > counts[0], "the sparse end must represent more features than the dense end"

At the dense end the model uses its five dimensions for five features and `WᵀW` is nearly
diagonal. At the sparse end it represents many more than five, and `WᵀW` fills in with
off-diagonal interference — the signature of superposition. Same mechanism as the
pentagon, now at scale.

## Recap, and what comes next

**What we proved.** Superposition is real: a ReLU model on sparse data represents more
features than it has dimensions (five in two, the pentagon; and many-in-five at scale).
Sparsity is the knob — dense data gives an orthogonal top-`m` basis (PCA-like), and only
as features get rare does the model superpose them. The cost is interference, visible as
off-diagonal mass in `WᵀW`.

**What we deferred — book two.** *Why* the directions pick a pentagon (and tetrahedrons,
digons, and other uniform polytopes) rather than any old non-orthogonal set; the full
phase diagram; "feature dimensionality" (the fraction of a dimension a feature gets); and
computation performed *in* superposition. Each is its own study, and each builds directly
on the toy we just made.